<a href="https://colab.research.google.com/github/amenidhawadi/deep-learning/blob/main/03_CNN_LSTM_tests_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import joblib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np



In [3]:
def cnn_1d(kernel=3, filters=64):
    model = keras.Sequential([
        layers.Input(shape=(X_train.shape[1],)),
        layers.Reshape((X_train.shape[1], 1)),
        layers.Conv1D(filters, kernel, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Conv1D(filters*2, kernel, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling1D(),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', 'AUC'])
    return model

In [4]:
def lstm_model(units=64):
    model = keras.Sequential([
        layers.Input(shape=(X_train.shape[1], 1)),
        layers.LSTM(units, return_sequences=True, dropout=0.3),
        layers.LSTM(units//2, dropout=0.3),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', 'AUC'])
    return model

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib

# Exemple : chargement de vos données brutes (CSV, Excel, etc.)
# df = pd.read_csv('votre_fichier.csv')
# Suppose que la colonne cible s'appelle 'target'
# X = df.drop('target', axis=1)
# y = df['target']

# Pour l’exemple, générons des données synthétiques
np.random.seed(42)
n_samples = 1000
n_features = 20
X = np.random.randn(n_samples, n_features)
y = (np.random.rand(n_samples) > 0.5).astype(int)  # classification binaire

# Division train/val/test (ex: 60/20/20)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Normalisation (StandardScaler)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Sauvegarde
joblib.dump((X_train, y_train, X_val, y_val, X_test, y_test, scaler), 'preprocessed_data.pkl')
print("Fichier 'preprocessed_data.pkl' créé avec succès.")

Fichier 'preprocessed_data.pkl' créé avec succès.


In [3]:
import os
print(os.listdir('.'))

['.config', 'preprocessed_data.pkl', 'sample_data']


In [4]:
data = joblib.load('/content/preprocessed_data.pkl')

In [5]:
import os
import joblib
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# ============================================
# 1. Chargement ou génération des données
# ============================================
data_file = 'preprocessed_data.pkl'

if os.path.exists(data_file):
    print("Chargement des données existantes...")
    X_train, y_train, X_val, y_val, X_test, y_test, scaler = joblib.load(data_file)
else:
    print("Fichier de données introuvable. Génération de données synthétiques...")
    # Paramètres synthétiques
    np.random.seed(42)
    n_samples = 2000
    n_features = 30
    X = np.random.randn(n_samples, n_features)
    y = (np.random.rand(n_samples) > 0.5).astype(int)

    # Split
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

    # Normalisation
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    # Sauvegarde pour les prochaines exécutions
    joblib.dump((X_train, y_train, X_val, y_val, X_test, y_test, scaler), data_file)
    print(f"Données synthétiques sauvegardées dans {data_file}")

# Convertir en DataFrame si nécessaire (pour compatibilité avec le code original)
# Ici X_train est déjà un numpy array, mais le code original utilisait .values.
# On peut le laisser ainsi ou le convertir en DataFrame.
# Pour coller au code original, créons des DataFrames factices :
import pandas as pd
X_train = pd.DataFrame(X_train)
X_val = pd.DataFrame(X_val)
X_test = pd.DataFrame(X_test)
y_train = pd.Series(y_train)
y_val = pd.Series(y_val)
y_test = pd.Series(y_test)

# ============================================
# 2. Préparation des données 3D pour CNN/LSTM
# ============================================
n_features = X_train.shape[1]
X_train_3d = X_train.values.reshape(-1, n_features, 1)
X_val_3d   = X_val.values.reshape(-1, n_features, 1)
X_test_3d  = X_test.values.reshape(-1, n_features, 1)

# ============================================
# 3. Définition des modèles
# ============================================
def cnn_1d(kernel=3, filters=64):
    model = keras.Sequential([
        layers.Input(shape=(n_features, 1)),
        layers.Conv1D(filters, kernel, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Conv1D(filters*2, kernel, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling1D(),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy',
                  metrics=['accuracy', 'AUC'])
    return model

def lstm_model(units=64):
    model = keras.Sequential([
        layers.Input(shape=(n_features, 1)),
        layers.LSTM(units, return_sequences=True, dropout=0.3),
        layers.LSTM(units//2, dropout=0.3),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy',
                  metrics=['accuracy', 'AUC'])
    return model

# ============================================
# 4. Entraînement et évaluation
# ============================================
print("\nEntraînement du CNN...")
cnn = cnn_1d()
history_cnn = cnn.fit(
    X_train_3d, y_train,
    validation_data=(X_val_3d, y_val),
    epochs=50, batch_size=32, verbose=1,
    callbacks=[keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)]
)

print("\nEntraînement du LSTM...")
lstm = lstm_model()
history_lstm = lstm.fit(
    X_train_3d, y_train,
    validation_data=(X_val_3d, y_val),
    epochs=50, batch_size=32, verbose=1,
    callbacks=[keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)]
)

print("\n=== Évaluation sur le test ===")
cnn_loss, cnn_acc, cnn_auc = cnn.evaluate(X_test_3d, y_test, verbose=0)
lstm_loss, lstm_acc, lstm_auc = lstm.evaluate(X_test_3d, y_test, verbose=0)

print(f"CNN  -> Accuracy : {cnn_acc:.4f}, AUC : {cnn_auc:.4f}")
print(f"LSTM -> Accuracy : {lstm_acc:.4f}, AUC : {lstm_auc:.4f}")

Chargement des données existantes...

Entraînement du CNN...
Epoch 1/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 4s 34ms/step - AUC: 0.5490 - accuracy: 0.5533 - loss: 0.7006 - val_AUC: 0.4890 - val_accuracy: 0.4800 - val_loss: 0.6936
Epoch 2/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - AUC: 0.5972 - accuracy: 0.5800 - loss: 0.6771 - val_AUC: 0.4783 - val_accuracy: 0.4950 - val_loss: 0.6934
Epoch 3/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - AUC: 0.6173 - accuracy: 0.5983 - loss: 0.6677 - val_AUC: 0.4452 - val_accuracy: 0.5400 - val_loss: 0.6929
Epoch 4/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - AUC: 0.6624 - accuracy: 0.6067 - loss: 0.6482 - val_AUC: 0.4403 - val_accuracy: 0.5400 - val_loss: 0.6934
Epoch 5/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - AUC: 0.6469 - accuracy: 0.6183 - loss: 0.6583 - val_AUC: 0.4400 - val_accuracy: 0.5350 - val_loss: 0.6934
Epoch 6/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - AUC: 0.6918 - accuracy: 0.6383 - loss: 0.6410 - val_AUC: 0.4596 - val_accuracy: 0.5400 - va